# TwinChain — Modelo de línea base v2 (mejoras sobre la guía)

Parte de `baseline_model.ipynb` (la guía, intacta). Las decisiones cerradas no se tocan:
Olist, `margin_days`, umbral de 60 días, Fase I = 4 semanas activas, las 7 features de la guía,
GradientBoostingRegressor con los mismos hiperparámetros, split 60/20/20 con semilla 42, CQR y **RF017**.

**Mejora adoptada: banda semanal.** La banda de la guía está calibrada para el *promedio* de toda la Fase II.
Si el monitoreo compara una *semana* contra esa banda, casi la mitad de las semanas cae afuera. Se agrega una
segunda banda, calibrada con CQR sobre valores semanales, que es la que corresponde usar para alertas semana a semana.

**Mejora evaluada y descartada: features extendidas de Fase I.** Son estadísticos propios (RF017 ok): mediana,
tendencia, distribución por ítem, fracción tardía y tiempo de preparación. Mejoran el MAE solo 0.02 días en splits
aleatorios y empeoran la cobertura en la validación temporal. Se documenta abajo por qué no se adoptan.

El cálculo de features vive en `features.py`, compartido con la API.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, joblib
from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor
from features import (load_items, weekly_from_items, build_seller_table, BASE_FEATURES, EXT_FEATURES,
                      PHASE1_N_WEEKS, MIN_PHASE1_WEEKS, MIN_PHASE2_WEEKS, OUTLIER_ABS_DAYS)

items = load_items('data')          # pasos 2-4 (+ prep_days para una feature)
feat, phase2_weeks = build_seller_table(items)   # pasos 5-6
print(f"Vendedores usables: {len(feat):,}")
feat[EXT_FEATURES + ['phase2_mean']].describe().round(2).T

Vendedores usables: 1,460


,count,mean,std,min,25%,50%,75%,max
phase1_mean,1460.0,3.36,2.80,-18.24,2.41,3.63,4.58,17.96
phase1_std,1460.0,1.97,2.15,0.05,0.96,1.41,2.07,25.38
phase1_min,1460.0,0.61,5.48,-58.59,0.03,1.87,2.95,14.09
phase1_max,1460.0,5.60,2.70,-3.95,4.47,5.36,6.11,32.92
phase1_median,1460.0,3.62,2.52,-16.43,2.51,3.70,4.78,20.09
phase1_mean_items_per_week,1460.0,1.98,1.61,1.00,1.25,1.50,2.25,26.50
phase1_total_items,1460.0,7.92,6.42,4.00,5.00,6.00,9.00,106.00
phase1_last,1460.0,3.45,3.94,-38.04,2.11,3.59,5.20,25.91
phase1_slope,1460.0,0.09,1.61,-12.30,-0.57,0.02,0.65,17.21
phase1_item_mean,1460.0,3.36,2.84,-20.27,2.39,3.64,4.54,20.95


## Split (idéntico a la guía: semilla 42, 60/20/20)

In [2]:
RNG_SEED = 42
QUANTILES = (0.1, 0.5, 0.9)
HP = dict(n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.8)

def split(df, seed):
    rng = np.random.RandomState(seed)
    idx = df.index.to_numpy().copy(); rng.shuffle(idx)
    n = len(idx); a, b = int(n * 0.6), int(n * 0.2)
    return (df.loc[idx[:a]].reset_index(drop=True), df.loc[idx[a:a + b]].reset_index(drop=True),
            df.loc[idx[a + b:]].reset_index(drop=True))

train_df, calib_df, test_df = split(feat, RNG_SEED)
print(f"Train: {len(train_df)} | Calibración: {len(calib_df)} | Test: {len(test_df)}")

Train: 876 | Calibración: 292 | Test: 292


## Funciones: entrenar, calibrar con CQR, evaluar
`cqr_Q` es exactamente el cálculo del Paso 10 de la guía. Para la banda semanal se usa la misma fórmula, pero
los puntos de calibración son las semanas de Fase II de los vendedores de calibración, en lugar de su promedio.

In [3]:
def fit_models(df, features, seed):
    return {q: GradientBoostingRegressor(loss='quantile', alpha=q, random_state=seed, **HP)
                 .fit(df[features].to_numpy(), df['phase2_mean'].to_numpy())
            for q in QUANTILES}

def cqr_Q(lo, hi, y, coverage=0.8):
    scores = np.maximum(lo - y, y - hi)
    n = len(scores)
    return np.quantile(scores, min(np.ceil((n + 1) * coverage) / n, 1.0), method='higher')

def weekly_rows(df):
    """Repite cada vendedor una vez por semana de su Fase II -> (fila de features, valor semanal)."""
    reps = df['seller_id'].map(lambda s: len(phase2_weeks[s])).to_numpy()
    y = np.concatenate([phase2_weeks[s] for s in df['seller_id']])
    return df.loc[df.index.repeat(reps)].reset_index(drop=True), y

def train_and_eval(train_df, calib_df, test_df, features, seed):
    models = fit_models(train_df, features, seed)
    P = lambda df, q: models[q].predict(df[features].to_numpy())
    # banda del promedio de Fase II (la de la guía)
    Q = cqr_Q(P(calib_df, 0.1), P(calib_df, 0.9), calib_df['phase2_mean'].to_numpy())
    # banda semanal (mejora adoptada)
    cw, cw_y = weekly_rows(calib_df)
    Q_week = cqr_Q(P(cw, 0.1), P(cw, 0.9), cw_y)

    y = test_df['phase2_mean'].to_numpy()
    lo_raw, hi_raw, med = P(test_df, 0.1), P(test_df, 0.9), P(test_df, 0.5)
    tw, tw_y = weekly_rows(test_df)
    wlo, whi = P(tw, 0.1) - Q_week, P(tw, 0.9) + Q_week
    wlo_old, whi_old = P(tw, 0.1) - Q, P(tw, 0.9) + Q
    r = {
        'MAE línea base': np.mean(np.abs(y - med)),
        'MAE ingenuo (media Fase I)': np.mean(np.abs(y - test_df['phase1_mean'])),
        'cobertura sin CQR': np.mean((y >= lo_raw) & (y <= hi_raw)),
        'cobertura CQR (promedio)': np.mean((y >= lo_raw - Q) & (y <= hi_raw + Q)),
        'ancho banda promedio': np.mean(hi_raw - lo_raw + 2 * Q),
        'cobertura semanal con banda del promedio': np.mean((tw_y >= wlo_old) & (tw_y <= whi_old)),
        'cobertura semanal con banda semanal': np.mean((tw_y >= wlo) & (tw_y <= whi)),
        'alarmas semanales (debajo de banda semanal)': np.mean(tw_y < wlo),
        'ancho banda semanal': np.mean(whi - wlo),
        'Q (días)': Q, 'Q semanal (días)': Q_week,
        'n_test': len(y), 'n_test_semanas': len(tw_y),
    }
    return models, Q, Q_week, r

## Guía (7 features) vs. features extendidas, mismo split (semilla 42)

In [4]:
models_v1, Q_v1, Qw_v1, r_v1 = train_and_eval(train_df, calib_df, test_df, BASE_FEATURES, RNG_SEED)
models_v2, Q_v2, Qw_v2, r_v2 = train_and_eval(train_df, calib_df, test_df, EXT_FEATURES, RNG_SEED)
comp = pd.DataFrame({'guía (7 features) — ADOPTADO': r_v1, 'features extendidas': r_v2})
comp.round(3)

,guía (7 features) — ADOPTADO,features extendidas
MAE línea base,1.358,1.341
MAE ingenuo (media Fase I),1.826,1.826
cobertura sin CQR,0.699,0.726
cobertura CQR (promedio),0.791,0.771
ancho banda promedio,3.814,3.576
cobertura semanal con banda del promedio,0.539,0.522
cobertura semanal con banda semanal,0.798,0.798
alarmas semanales (debajo de banda semanal),0.077,0.078
ancho banda semanal,5.965,5.729
Q (días),0.299,0.254


In [5]:
n = r_v2['n_test']
for name, r in [('guía', r_v1), ('features extendidas', r_v2)]:
    p = stats.binomtest(int(round(r['cobertura CQR (promedio)'] * n)), n, 0.8).pvalue
    print(f"{name}: cobertura CQR {r['cobertura CQR (promedio)']:.1%}, p-value vs 80% = {p:.3f} "
          f"({'sí' if p < 0.05 else 'no'} es significativamente distinta)")

guía: cobertura CQR 79.1%, p-value vs 80% = 0.714 (no es significativamente distinta)
features extendidas: cobertura CQR 77.1%, p-value vs 80% = 0.214 (no es significativamente distinta)


## Robustez 1: 20 particiones distintas
Un solo split de 292 vendedores es ruidoso (el MAE varía ±0.13 entre splits). Comparamos guía vs. features extendidas *pareado*,
sobre las mismas 20 particiones.

In [6]:
rows = []
for seed in range(20):
    tr, ca, te = split(feat, seed)
    _, _, _, a = train_and_eval(tr, ca, te, BASE_FEATURES, seed)
    _, _, _, b = train_and_eval(tr, ca, te, EXT_FEATURES, seed)
    rows.append({'seed': seed, 'MAE guía': a['MAE línea base'], 'MAE ext': b['MAE línea base'],
                 'ancho guía': a['ancho banda promedio'], 'ancho ext': b['ancho banda promedio'],
                 'cob guía': a['cobertura CQR (promedio)'], 'cob ext': b['cobertura CQR (promedio)'],
                 'cob semanal guía': a['cobertura semanal con banda semanal']})
rob = pd.DataFrame(rows)
d = rob['MAE ext'] - rob['MAE guía']
print(rob.drop(columns='seed').agg(['mean', 'std']).round(3).to_string())
print(f"\nfeatures extendidas mejoran el MAE en {np.mean(d < 0):.0%} de los splits; diferencia media {d.mean():+.3f} días "
      f"(Wilcoxon p = {stats.wilcoxon(d).pvalue:.4f})")

      MAE guía  MAE ext  ancho guía  ancho ext  cob guía  cob ext  cob semanal guía
mean     1.276    1.257       3.748      3.720     0.800    0.803             0.800
std      0.087    0.084       0.214      0.255     0.034    0.037             0.022

features extendidas mejoran el MAE en 80% de los splits; diferencia media -0.019 días (Wilcoxon p = 0.0056)


## Robustez 2: validación temporal
El split aleatorio mezcla vendedores de distintas épocas. Acá entrenamos con los vendedores cuya Fase I terminó
primero y testeamos con los más nuevos, que es como se usaría en producción.

In [7]:
tfeat = feat.sort_values('phase1_end_week').reset_index(drop=True)
n = len(tfeat); a, b = int(n * 0.6), int(n * 0.8)
ttr, tca, tte = tfeat.iloc[:a].reset_index(drop=True), tfeat.iloc[a:b].reset_index(drop=True), tfeat.iloc[b:].reset_index(drop=True)
print(f"train hasta {ttr.phase1_end_week.max().date()}, test desde {tte.phase1_end_week.min().date()}")
_, _, _, ta = train_and_eval(ttr, tca, tte, BASE_FEATURES, RNG_SEED)
_, _, _, tb = train_and_eval(ttr, tca, tte, EXT_FEATURES, RNG_SEED)
pd.DataFrame({'guía': ta, 'features extendidas': tb}).round(3)

train hasta 2017-12-18, test desde 2018-04-16


,guía,features extendidas
MAE línea base,1.331,1.334
MAE ingenuo (media Fase I),1.839,1.839
cobertura sin CQR,0.620,0.541
cobertura CQR (promedio),0.658,0.592
ancho banda promedio,3.118,2.750
cobertura semanal con banda del promedio,0.403,0.355
cobertura semanal con banda semanal,0.791,0.763
alarmas semanales (debajo de banda semanal),0.144,0.166
ancho banda semanal,5.810,5.547
Q (días),0.106,0.115


### Por qué cae la cobertura de la banda del promedio en la validación temporal
El dataset termina en 2018: los vendedores más nuevos tienen pocas semanas de Fase II, así que su *promedio* es muy
ruidoso. La banda semanal no depende de eso, y por eso es la que se usa para monitorear.

In [8]:
m1 = fit_models(ttr, BASE_FEATURES, RNG_SEED)
P = lambda d, q: m1[q].predict(d[BASE_FEATURES].to_numpy())
y = tte['phase2_mean'].to_numpy()
Q_t = cqr_Q(P(tca, 0.1), P(tca, 0.9), tca['phase2_mean'].to_numpy())
inb = (y >= P(tte, 0.1) - Q_t) & (y <= P(tte, 0.9) + Q_t)
largo = tte['phase2_n_weeks'].to_numpy() >= 8
print(f"Semanas de Fase II (mediana) — train: {ttr.phase2_n_weeks.median():.0f}, test: {tte.phase2_n_weeks.median():.0f}")
print(f"Cobertura banda del promedio, test temporal: Fase II >= 8 semanas {inb[largo].mean():.1%} (n={largo.sum()}) | "
      f"< 8 semanas {inb[~largo].mean():.1%} (n={(~largo).sum()})")

Semanas de Fase II (mediana) — train: 19, test: 4
Cobertura banda del promedio, test temporal: Fase II >= 8 semanas 83.7% (n=49) | < 8 semanas 62.1% (n=243)


## Guardar el modelo v2 (7 features de la guía + banda semanal)
Se guarda aparte (`baseline_model_bundle_v2.joblib`); el bundle de la guía queda intacto.

In [9]:
bundle = {
    'version': 'v2',
    'model_family': 'GradientBoostingRegressor (scikit-learn)',
    'uncertainty_technique': 'Conformalized Quantile Regression (CQR)',
    'quantile_models': models_v1,
    'cqr_adjustment_Q_days': Q_v1,
    'cqr_adjustment_Q_weekly_days': Qw_v1,
    'quantiles': {'lo': 0.1, 'median': 0.5, 'hi': 0.9},
    'feature_columns': BASE_FEATURES,
    'phase1_n_weeks': PHASE1_N_WEEKS,
    'min_phase1_weeks': MIN_PHASE1_WEEKS,
    'min_phase2_weeks': MIN_PHASE2_WEEKS,
    'outlier_abs_days_threshold': OUTLIER_ABS_DAYS,
    'training_dataset': 'Olist (olist_orders_dataset.csv + olist_order_items_dataset.csv)',
    'target_definition': (
        'margin_days = shipping_limit_date - order_delivered_carrier_date (dias); '
        'positivo=despacho antes del compromiso propio del vendedor, negativo=despacho tardio.'
    ),
    'bands': {
        'banda_promedio': 'cubre el 80% del promedio de Fase II del vendedor',
        'banda_semanal': 'cubre el 80% de los valores semanales de Fase II (usar para alertas semanales)',
    },
    'sklearn_hyperparameters': {**HP, 'random_state': RNG_SEED},
    'test_metrics_seed42': r_v1,
    'temporal_validation': ta,
}
joblib.dump(bundle, 'baseline_model_bundle_v2.joblib')
print("Modelo guardado en baseline_model_bundle_v2.joblib")

Modelo guardado en baseline_model_bundle_v2.joblib
